**Silver Layer: Sales Data Transformation**

This notebook merges yearly sales data (2017-2020) from the Bronze layer, performs data cleansing, and handles schema evolution issues (e.g., the introduction of the 'discount' column in 2020).

**Simulating Data Inconsistencies (Sabotage for Testing)**
To demonstrate the robustness of our Silver Layer transformation, we are intentionally introducing common "Legacy System" issues into the **2020 Bronze dataset**. 

**Scenario Objectives:**
1. **Schema Evolution:** Add a new `discount` column that only exists in 2020.
2. **Date Inconsistency:** Change the format of specific rows to a standard ISO format (`YYYY-MM-DD`) to test multi-format parsing.
3. **Data Integrity:** Remove a `product_key` to test how the pipeline handles missing foreign keys.

In [0]:
from pyspark.sql import functions as F

# Load the original 2020 Bronze table
df_2020_raw = spark.table("bronze.bronze_sales_2020")

print("Original Schema Check (2020):")
df_2020_raw.printSchema()

# --- STEP 1: Introduce Schema Evolution (The 'discount' column) ---
# We simulate a business change where discounts started being recorded in 2020.
# We apply a $5.00 discount only for orders with quantity > 5.
df_2020_sabotaged = df_2020_raw.withColumn("discount", 
    F.when(F.col("quantity") > 5, F.lit("$5.00")).otherwise(F.lit(None))
)

# --- STEP 2: Introduce Date Inconsistency ---
# We change a specific order's date to '2020-05-15' (ISO format) 
# to break the 'Thursday, February 27, 2020' pattern.
target_order = "SO63283"
df_2020_sabotaged = df_2020_sabotaged.withColumn("order_date", 
    F.when(F.col("sales_order_number") == target_order, F.lit("2020-05-15"))
     .otherwise(F.col("order_date"))
)

# --- STEP 3: Introduce Missing Keys (Null Product Key) ---
# We nullify the product_key for the same order to test Referential Integrity handling.
df_2020_sabotaged = df_2020_sabotaged.withColumn("product_key", 
    F.when(F.col("sales_order_number") == target_order, F.lit(None))
     .otherwise(F.col("product_key"))
)

# --- STEP 4: Overwrite the Bronze Table ---
# We overwrite the table so the Silver Layer script "sees" the dirty data.
df_2020_sabotaged.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze.bronze_sales_2020")

print("-" * 30)
print("SUCCESS: 2020 Bronze data has been sabotaged for testing purposes.")
print("New Column 'discount' added. Inconsistent dates and null keys injected.")

###Silver Transformation & Refactoring

This script executes the unified cleansing and transformation logic for the Sales dataset (2017-2020). It has been refactored to ensure high data quality and system robustness.

**Key Operations:**
* **Schema Evolution:** Merging datasets with dynamic column support (e.g., handling the `discount` column from 2020).
* **Advanced Date Parsing:** Using `coalesce` to handle inconsistent date formats (Long-form vs ISO).
* **Financial Precision:** Implementing `Decimal(18,2)` types for all monetary values to ensure accounting accuracy.
* **Data Integrity:** Imputing missing `product_key` entries with `-1` and filling null discounts with `0.00`.
* **Corrected Deduplication:** Preservation of all line items within orders by applying global row deduplication.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

# --- Step 1: Data Ingestion & Merging ---
s17 = spark.table("bronze.bronze_sales_2017")
s18 = spark.table("bronze.bronze_sales_2018")
s19 = spark.table("bronze.bronze_sales_2019")
s20 = spark.table("bronze.bronze_sales_2020")

df = s17.unionByName(s18, allowMissingColumns=True) \
        .unionByName(s19, allowMissingColumns=True) \
        .unionByName(s20, allowMissingColumns=True)

# --- Step 2: Multi-Currency Detection ---
df = df.withColumn("currency_code", 
    F.when(F.col("sales").rlike(r"\$"), "USD")
     .when(F.col("sales").rlike("€"), "EUR")
     .when(F.col("sales").rlike("£"), "GBP")
     .otherwise("USD")
)

# --- Step 3: Universal String Cleansing ---
string_cols = [c for c, t in df.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(c, 
        F.when(F.lower(cleaned).isin("n/a", "na", "null", "none", "-"), None)
         .otherwise(cleaned)
    )

# --- Step 4: Robust Date Parsing (Spark 3.x Fix) ---
#  Instead of coalesce (which crashes on mismatch in Spark 3), 
# we use 'when' to check the pattern before parsing.
df = df.withColumn("order_date", 
    F.when(F.col("order_date").rlike(r"^\d{4}-\d{2}-\d{2}$"), 
           F.to_date(F.col("order_date"), "yyyy-MM-dd"))
     .otherwise(
           F.to_date(F.regexp_replace(F.col("order_date"), r"^[A-Za-z]+,\s*", ""), "MMMM d, yyyy")
     )
)

# --- Step 5: Financial Precision ---
money_mapping = {
    "unit_price": "unit_price_amount",
    "sales": "sales_amount",
    "cost": "cost_amount",
    "discount": "discount_amount"
}

for old_name, new_name in money_mapping.items():
    if old_name in df.columns:
        df = df.withColumn(new_name, 
            F.regexp_replace(F.col(old_name), r"[\$,€,£]", "").cast("decimal(18,2)")
        )
        if old_name != new_name:
            df = df.drop(old_name)

# --- Step 6: Optimization (Integer Casting) ---
int_cols = ["product_key", "reseller_key", "employee_key", "sales_territory_key", "quantity"]
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast("int"))

# --- Step 7: Data Integrity & Imputation ---
if "discount_amount" in df.columns:
    df = df.fillna(0, subset=["discount_amount"])

df = df.fillna(-1, subset=["product_key"])

# --- Step 8: Deduplication & Quality Filters ---
df = df.dropDuplicates()
df = df.filter(F.col("quantity") > 0)

# --- Step 9: Metadata & Save ---
df = df.withColumn("silver_processed_at", F.current_timestamp())

df.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .format("delta") \
    .saveAsTable("silver.silver_sales")

print("Pipeline Status: Success. Silver Table 'silver_sales' updated.")
display(df.limit(10))